This notebook contains the code for scaling the preprocessing to all the subject done earlier to subject S2. 

The result of this pipeline is stored in ../data/processed.

In [1]:
import pickle
import numpy as np
import os
from scipy.signal import butter, sosfiltfilt, savgol_filter

Defining reusable preprocessing functions

def bandpass(signal, low, high, fs, order=4): - To apply Butterworth bandpass filter

def zscore(signal): - To apply Z-score normalization

def preprocess_subject(pkl_path, fs=700, window_sec=30, stride_sec=15): -Complete preprocessing pipeline for a single WESAD subject

In [ ]:
def bandpass(signal, low, high, fs, order=4):
    nyquist = fs / 2
    sos = butter(order, [low / nyquist, high / nyquist], btype="band", output="sos")
    return sosfiltfilt(sos, signal)

def zscore(signal):
    return (signal - np.mean(signal)) / (np.std(signal) + 1e-8)

def preprocess_subject(pkl_path, fs=700, window_sec=30, stride_sec=15):

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    chest  = data["signal"]["chest"]
    labels = data["label"]

    ecg  = chest["ECG"].flatten()
    eda  = chest["EDA"].flatten()
    resp = chest["Resp"].flatten()
    temp = chest["Temp"].flatten()

    mask   = np.isin(labels, [1, 2, 3])
    ecg    = ecg[mask]
    eda    = eda[mask]
    resp   = resp[mask]
    temp   = temp[mask]
    labels = labels[mask]

    labels = np.where(labels == 2, 1, 0)

    ecg_f  = bandpass(ecg,  low=0.5,  high=40.0, fs=fs)
    eda_f  = bandpass(eda,  low=0.05, high=5.0,  fs=fs)
    resp_f = bandpass(resp, low=0.05, high=1.0,  fs=fs)
    temp_f = savgol_filter(temp, window_length=701, polyorder=2)

    ecg_n  = zscore(ecg_f)
    eda_n  = zscore(eda_f)
    resp_n = zscore(resp_f)
    temp_n = zscore(temp_f)

    window_size = window_sec * fs   
    stride      = stride_sec * fs   

    X, y = [], []
    for start in range(0, len(labels) - window_size, stride):
        end = start + window_size

        sample = np.stack([
            ecg_n[start:end],
            eda_n[start:end],
            resp_n[start:end],
            temp_n[start:end]
        ])

        window_label = np.bincount(labels[start:end]).argmax()

        X.append(sample)
        y.append(window_label)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    return X, y

Apply the pipeline to all subjects from 2 to 17

The subjects are processsed sequentially and the outputs are saved immediately.

In [ ]:
SUBJECTS   = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
RAW_DIR    = "../data/raw"
SAVE_DIR   = "../data/processed"

os.makedirs(SAVE_DIR, exist_ok=True)

summary = []   

for sid in SUBJECTS:
    pkl_path = os.path.join(RAW_DIR, f"S{sid}", f"S{sid}.pkl")

    # Skip if the file doesn't exist (some WESAD subject data are missing subjects)
    if not os.path.exists(pkl_path):
        print(f"S{sid}: file not found — skipping")
        continue

    print(f"Processing S{sid}...", end=" ")

    X, y = preprocess_subject(pkl_path)

    # Verify no NaNs before saving
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        print(f"WARNING — {nan_count} NaNs found, skipping S{sid}")
        continue

    # Save
    np.save(os.path.join(SAVE_DIR, f"S{sid}_X.npy"), X)
    np.save(os.path.join(SAVE_DIR, f"S{sid}_y.npy"), y)

    stress     = (y == 1).sum()
    nonstress  = (y == 0).sum()
    total      = len(y)

    summary.append({
        "subject"   : f"S{sid}",
        "total"     : total,
        "stress"    : stress,
        "nonstress" : nonstress,
        "stress_pct": round(stress / total * 100, 1)
    })

    print(f"done — shape {X.shape}  stress={stress}  non-stress={nonstress}")

print("\nAll subjects complete.")

Processing S2... done — shape (140, 4, 21000)  stress=41  non-stress=99
Processing S3... done — shape (142, 4, 21000)  stress=42  non-stress=100
Processing S4... done — shape (143, 4, 21000)  stress=41  non-stress=102
Processing S5... done — shape (146, 4, 21000)  stress=42  non-stress=104
Processing S6... done — shape (145, 4, 21000)  stress=43  non-stress=102
Processing S7... done — shape (145, 4, 21000)  stress=42  non-stress=103
Processing S8... done — shape (146, 4, 21000)  stress=44  non-stress=102
Processing S9... done — shape (145, 4, 21000)  stress=43  non-stress=102
Processing S10... done — shape (150, 4, 21000)  stress=47  non-stress=103
Processing S11... done — shape (147, 4, 21000)  stress=45  non-stress=102
S12: file not found — skipping
Processing S13... done — shape (147, 4, 21000)  stress=43  non-stress=104
Processing S14... done — shape (147, 4, 21000)  stress=45  non-stress=102
Processing S15... done — shape (147, 4, 21000)  stress=44  non-stress=103
Processing S16..

In [4]:
print(f"\n{'Subject':<10} {'Total':>7} {'Stress':>8} {'Non-stress':>12} {'Stress %':>10}")
print("-" * 52)
for row in summary:
    print(f"{row['subject']:<10} {row['total']:>7} {row['stress']:>8} {row['nonstress']:>12} {row['stress_pct']:>9.1f}%")

total_windows  = sum(r["total"]   for r in summary)
total_stress   = sum(r["stress"]  for r in summary)
total_ns       = sum(r["nonstress"] for r in summary)

print("-" * 52)
print(f"{'TOTAL':<10} {total_windows:>7} {total_stress:>8} {total_ns:>12} {total_stress/total_windows*100:>9.1f}%")


Subject      Total   Stress   Non-stress   Stress %
----------------------------------------------------
S2             140       41           99      29.3%
S3             142       42          100      29.6%
S4             143       41          102      28.7%
S5             146       42          104      28.8%
S6             145       43          102      29.7%
S7             145       42          103      29.0%
S8             146       44          102      30.1%
S9             145       43          102      29.7%
S10            150       47          103      31.3%
S11            147       45          102      30.6%
S13            147       43          104      29.3%
S14            147       45          102      30.6%
S15            147       44          103      29.9%
S16            147       45          102      30.6%
S17            150       47          103      31.3%
----------------------------------------------------
TOTAL         2187      654         1533      29.9%


Verifying if all the files exist on the disk

In [5]:
print("Files in processed/:")
missing = []
for sid in SUBJECTS:
    x_path = os.path.join(SAVE_DIR, f"S{sid}_X.npy")
    y_path = os.path.join(SAVE_DIR, f"S{sid}_y.npy")
    x_ok = os.path.exists(x_path)
    y_ok = os.path.exists(y_path)
    status = "OK" if (x_ok and y_ok) else "MISSING"
    if status == "MISSING":
        missing.append(f"S{sid}")
    print(f"  S{sid}_X.npy: {'✓' if x_ok else '✗'}    S{sid}_y.npy: {'✓' if y_ok else '✗'}")

if missing:
    print(f"\nMissing subjects: {missing}")
else:
    print("\nAll files present.")

Files in processed/:
  S2_X.npy: ✓    S2_y.npy: ✓
  S3_X.npy: ✓    S3_y.npy: ✓
  S4_X.npy: ✓    S4_y.npy: ✓
  S5_X.npy: ✓    S5_y.npy: ✓
  S6_X.npy: ✓    S6_y.npy: ✓
  S7_X.npy: ✓    S7_y.npy: ✓
  S8_X.npy: ✓    S8_y.npy: ✓
  S9_X.npy: ✓    S9_y.npy: ✓
  S10_X.npy: ✓    S10_y.npy: ✓
  S11_X.npy: ✓    S11_y.npy: ✓
  S12_X.npy: ✗    S12_y.npy: ✗
  S13_X.npy: ✓    S13_y.npy: ✓
  S14_X.npy: ✓    S14_y.npy: ✓
  S15_X.npy: ✓    S15_y.npy: ✓
  S16_X.npy: ✓    S16_y.npy: ✓
  S17_X.npy: ✓    S17_y.npy: ✓

Missing subjects: ['S12']


Verifying if all the shapes are consistent

In [ ]:
print(f"{'Subject':<10} {'X shape':<20} {'y shape':<12}")
print("-" * 45)

for sid in SUBJECTS:
    x_path = os.path.join(SAVE_DIR, f"S{sid}_X.npy")
    if not os.path.exists(x_path):
        continue
    X = np.load(x_path)
    y = np.load(os.path.join(SAVE_DIR, f"S{sid}_y.npy"))
    shape_ok = (X.ndim == 3 and X.shape[1] == 4 and X.shape[2] == 21000)
    flag = "" if shape_ok else "  ← CHECK THIS"
    print(f"  S{sid:<8} {str(X.shape):<20} {str(y.shape):<12}{flag}")

print("\nDone. All processed data verified.")

Subject    X shape              y shape     
---------------------------------------------
  S2        (140, 4, 21000)      (140,)      
  S3        (142, 4, 21000)      (142,)      
  S4        (143, 4, 21000)      (143,)      
  S5        (146, 4, 21000)      (146,)      
  S6        (145, 4, 21000)      (145,)      
  S7        (145, 4, 21000)      (145,)      
  S8        (146, 4, 21000)      (146,)      
  S9        (145, 4, 21000)      (145,)      
  S10       (150, 4, 21000)      (150,)      
  S11       (147, 4, 21000)      (147,)      
  S13       (147, 4, 21000)      (147,)      
  S14       (147, 4, 21000)      (147,)      
  S15       (147, 4, 21000)      (147,)      
  S16       (147, 4, 21000)      (147,)      
  S17       (150, 4, 21000)      (150,)      

Done. All processed data verified.
